# Landslide Prediction using XGBoost


## Importing the Required Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    average_precision_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    cohen_kappa_score,
    ConfusionMatrixDisplay,
    roc_auc_score
)


## Loading the Training and Test Data


In [ ]:
train = pd.read_csv("landslide_train_feature_engineered.csv")
test = pd.read_csv("landslide_test_feature_engineered.csv")


In [ ]:
print("Training shape:", train.shape)
print("Test shape:", test.shape)


In [ ]:
print(train.head())


## Preparing Features and Target


In [ ]:
X_train = train.drop(columns=["class"])
y_train = train["class"]


In [ ]:
X_test = test.drop(columns=["class"])
y_test = test["class"]


In [ ]:
assert list(X_train.columns) == list(X_test.columns)


In [ ]:
print("Number of input features:", X_train.shape[1])
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


## Creating the XGBoost Model


In [ ]:
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)


## Training the Model


In [ ]:
xgb_model.fit(X_train, y_train)


In [ ]:
print("XGBoost training completed!")


## Making Predictions on Test Data


In [ ]:
y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:, 1]

print("Predictions completed!")
print("First 10 predictions:", y_pred[:10])


## Checking Model Performance


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()


In [ ]:
results = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall / Sensitivity": recall_score(y_test, y_pred, zero_division=0),
    "Specificity": tn / (tn + fp) if (tn + fp) else 0,
    "F1-score": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob),
    "PR-AUC (Average Precision)": average_precision_score(y_test, y_prob),
    "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
    "MCC": matthews_corrcoef(y_test, y_pred),
    "Cohen's Kappa": cohen_kappa_score(y_test, y_pred)
}


In [ ]:
print("XGBoost — Evaluation Results")
print("-" * 42)
for metric, value in results.items():
    print(f"{metric}: {value:.4f}")

print("\nDetailed Classification Report:")
print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["Non-landslide", "Landslide"],
    zero_division=0
))

print("Confusion Matrix:")
print(cm)


## Viewing the Confusion Matrix


In [ ]:
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Non-landslide", "Landslide"]
)


In [ ]:
disp.plot()
plt.title("XGBoost - Landslide Classification")
plt.show()


## Checking Important Features


In [ ]:
importance = pd.Series(
    xgb_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importance.head(15))


In [ ]:
importance.head(15).sort_values().plot(
    kind="barh",
    figsize=(9, 6)
)

plt.title("Top 15 XGBoost Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


## Saving the Trained Model


In [ ]:
import joblib

joblib.dump(xgb_model, "xgboost_landslide_model.pkl")
print("Model saved as xgboost_landslide_model.pkl")


## Saving Test Predictions to CSV


In [ ]:
# Store actual labels, model predictions and landslide probabilities
predictions_df = pd.DataFrame({
    "Actual": y_test.reset_index(drop=True),
    "Predicted": y_pred,
    "Landslide_Probability": y_prob
})

# Save the prediction results
predictions_df.to_csv("xgboost_landslide_predictions.csv", index=False)

print("Predictions saved as xgboost_landslide_predictions.csv")
predictions_df.head(10)
